# Desktop Voice Orchestrator — M4 本地 3B/4B 路由器精调与 GGUF 导出流水线 (`dvo-router`, Colab A100 加速版)

本 Notebook 在 **Google Colab (推荐 A100 GPU)** 上一键完成 M4 路由器蒸馏微调与量化导出全流程，并内置 **`/content/local_router` 本地 NVMe 高速暂存 + 云盘增量断点同步**：
1. **Cell 1**：挂载 Google Drive，检查 A100 GPU 显存，拉取最新 `develop` 分支代码，初始化本地 NVMe 高速暂存盘 `/content/local_router`；
2. **Cell 2**：安装环境依赖（`peft`, `accelerate`, `transformers`, `datasets`, `sentencepiece`, `protobuf`, `gguf`, `tiktoken`）并配置转换环境；
3. **Cell 3（教师蒸馏与鲁棒性扩充）**：优先使用 `google.colab.userdata` 中的 `GEMINI_API_KEY` 或 `OPENAI_API_KEY` + 离线语音拼音近音噪声合成 **3,000 条** 严格符合规范的 SFT 训练集与验证集（自动备份至云盘）；
4. **Cell 4（基线零样本评测）**：加载 `Qwen/Qwen3-4B`（显式禁用思维链 `enable_thinking=False` 保证零思考延迟，并支持一键切换 `Qwen/Qwen2.5-3B-Instruct`），在验证集上摸底基线准确率与负样本误触发率；
5. **Cell 5（A100 bf16 LoRA SFT 精调）**：采用原生 `bfloat16 + sdpa` (`r=16, lora_alpha=32`) 与标准 `transformers.Trainer`、仅计算 Assistant 回复损失（Completion-only loss）、余弦学习率衰减，训练过程增量同步 Checkpoint 至云盘；
6. **Cell 6（对比评测、GGUF Q4_K_M 导出与交付）**：运行微调后验证集评测，合并权重并导出 `dvo-qwen3-4b-router-q4_k_m.gguf` 与 `Modelfile.dvo-router` 到云盘 `export/` 目录，供主力机 RTX 4090 极速加载。

## Cell 1：挂载 Google Drive、硬件体检并初始化 `/content/local_router` 高速盘

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

# =============================================================================
# 运行时控制开关
# =============================================================================
ENABLE_REMOTE_RESCUE = True
AUTO_DISCONNECT_A100 = True

drive.mount("/content/drive")

DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/desktop-voice-orchestrator-data")
DRIVE_ROOT = DRIVE_DATA_ROOT / "router_m4"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_ROUTER = Path("/content/local_router")
LOCAL_ROUTER.mkdir(parents=True, exist_ok=True)
(LOCAL_ROUTER / "dataset").mkdir(exist_ok=True)
(LOCAL_ROUTER / "checkpoints").mkdir(exist_ok=True)
(LOCAL_ROUTER / "export").mkdir(exist_ok=True)

REPO_DIR = Path("/content/desktop-voice-orchestrator")
REPO_URL = "https://github.com/CICSilver/desktop-voice-orchestrator.git"
REPO_BRANCH = "develop"

print("=== 1. 准备最新训练脚本仓库 (develop 分支) ===")
if not REPO_DIR.is_dir():
    print(f"正在克隆仓库 {REPO_URL} ({REPO_BRANCH}) ...")
    rc = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]).returncode
    if rc != 0:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Overlay repo_sync hotfixes if present on Drive
DRIVE_REPO_SYNC = DRIVE_DATA_ROOT / "repo_sync"
if DRIVE_REPO_SYNC.is_dir():
    print(f"🔄 检测到云盘热补丁目录: {DRIVE_REPO_SYNC}，正在覆盖同步到本地 NVMe {REPO_DIR} ...")
    subprocess.run(["rsync", "-a", f"{DRIVE_REPO_SYNC}/", f"{REPO_DIR}/"], check=True)
    print(f"✅ repo_sync 热补丁已成功覆盖应用到: {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"工作区路径: {REPO_DIR}")

from training.colab_telemetry_hook import ColabJobHook, make_hf_trainer_callback

hook = ColabJobHook(
    "m4_router",
    drive_data_root=DRIVE_DATA_ROOT,
    repo_dir=REPO_DIR,
    total_stages=6,
    rescue_wait_sec=900.0 if ENABLE_REMOTE_RESCUE else 0.0,
    auto_disconnect=AUTO_DISCONNECT_A100,
)

def stage1_setup_and_init():
    print("=== GPU 硬件检查 ===")
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    print(gpu_info.stdout.strip() if gpu_info.returncode == 0 else "⚠️ 未检测到 GPU，请在运行时设置中切换为 A100（或 L4）！")

    print("\n=== 增量断点恢复检查 ===")
    def sync_from_drive() -> None:
        for sub in ["dataset", "checkpoints", "export"]:
            src = DRIVE_ROOT / sub
            dst = LOCAL_ROUTER / sub
            if src.is_dir():
                subprocess.run(["rsync", "-a", f"{src}/", str(dst)], check=False)

    sync_from_drive()
    print("✅ 本地 NVMe 高速盘就绪，云盘历史断点检查完毕。")
    hook.update_progress(message="stage1_completed", force_flush=True)

hook.run_stage("cell1_setup_and_init", 1, stage1_setup_and_init)


## Cell 2：一键安装 PEFT、Accelerate、Transformers 与 GGUF 导出工具链

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

LLAMA_CPP_DIR = Path("/content/llama.cpp")

def stage2_install_deps():
    print("=== 1. 安装核心精调库 (PEFT, Accelerate, Transformers, Datasets) ===")
    pkgs = [
        "peft>=0.14.0",
        "accelerate>=1.2.0",
        "transformers>=4.51.0",
        "datasets",
        "sentencepiece",
        "protobuf>=5.28.3,<7.0.0",
        "gguf",
        "tiktoken",
    ]
    hook.run_cmd(["pip", "install", "-q", *pkgs], stage_name="cell2_pip_install")

    print("\n=== 2. 准备 llama.cpp 转换与量化工具 ===")
    if not (LLAMA_CPP_DIR / "convert_hf_to_gguf.py").is_file():
        print("克隆 llama.cpp 仓库以获取 GGUF 转换器...")
        hook.run_cmd(["git", "clone", "--depth=1", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)], stage_name="cell2_llamacpp_clone")

    quantize_built = (LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize").is_file() or (LLAMA_CPP_DIR / "llama-quantize").is_file()
    if not quantize_built:
        # 编译 llama-quantize (现代 llama.cpp 使用 CMake 构建)
        hook.run_cmd(["cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(LLAMA_CPP_DIR / "build"), "-DGGML_CUDA=OFF"], stage_name="cell2_llamacpp_cmake_cfg")
        hook.run_cmd(["cmake", "--build", str(LLAMA_CPP_DIR / "build"), "--config", "Release", "--target", "llama-quantize", "-j"], stage_name="cell2_llamacpp_build")

    # 防御性检查：若 torchvision 与当前 torch 二进制失配（如 torchvision::nms 缺失），直接卸载以避免污染 transformers 导入
    tv_check = subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True, text=True)
    if tv_check.returncode != 0:
        print("⚠️ 检测到 torchvision 二进制兼容异常，正在卸载失配的 torchvision / torchaudio...")
        hook.run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], stage_name="cell2_fix_torchvision")

    # 防御性检查：Colab 预装 torchao 0.10.0 与新版 peft (>=0.16.0 要求) 冲突，若不兼容直接卸载
    tao_check = subprocess.run([sys.executable, "-c", "from peft.import_utils import is_torchao_available; is_torchao_available()"], capture_output=True, text=True)
    if tao_check.returncode != 0:
        print("⚠️ 检测到不兼容的预装 torchao 版本，正在卸载以避免 peft LoRA 注入报错...")
        hook.run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], stage_name="cell2_fix_torchao")

    print("✅ 训练与导出依赖环境配置完毕！")
    hook.update_progress(message="cell2_completed", force_flush=True)

hook.run_stage("cell2_install_deps", 2, stage2_install_deps)


## Cell 3：教师模型蒸馏与离线规则/拼音近音噪声数据合成

In [ ]:
import json
import os
import subprocess
from pathlib import Path

DATASET_LOCAL = LOCAL_ROUTER / "dataset"
TRAIN_JSONL = DATASET_LOCAL / "train.jsonl"
VAL_JSONL = DATASET_LOCAL / "val.jsonl"
GEN_SCRIPT = REPO_DIR / "training" / "router" / "generate_router_dataset.py"

def stage3_distill_and_generate_dataset():
    print("=== 1. 检测 API Key 配置 (可选，支持无 Key 离线高保真合成) ===")
    try:
        from google.colab import userdata
        gemini_key = userdata.get('GEMINI_API_KEY')
        openai_key = userdata.get('OPENAI_API_KEY')
    except Exception:
        gemini_key = os.environ.get('GEMINI_API_KEY')
        openai_key = os.environ.get('OPENAI_API_KEY')

    if gemini_key:
        os.environ['GEMINI_API_KEY'] = gemini_key
        print("✅ 已配置 GEMINI_API_KEY，启用 Gemini 教师模型蒸馏链路。")
    elif openai_key:
        os.environ['OPENAI_API_KEY'] = openai_key
        print("✅ 已配置 OPENAI_API_KEY，启用 GPT 教师模型蒸馏链路。")
    else:
        print("ℹ️ 未检测到云端 API Key，将采用内置高保真规则与中文 ASR 拼音同音词噪声注入生成。")

    assert GEN_SCRIPT.is_file(), f"未找到数据集生成脚本: {GEN_SCRIPT}"

    print("\n=== 2. 运行自检并生成 3,000 条训练与验证样本 ===")
    hook.run_cmd(["python", str(GEN_SCRIPT), "--self-test"], stage_name="cell3_dataset_selftest")
    hook.run_cmd([
        "python", str(GEN_SCRIPT),
        "--output-dir", str(DATASET_LOCAL),
        "--num-samples", "3000",
        "--train-ratio", "0.9",
        "--seed", "42",
    ], stage_name="cell3_dataset_generate")

    print("\n=== 3. 同步数据集备份至 Google Drive ===")
    DRIVE_DATASET = DRIVE_ROOT / "dataset"
    DRIVE_DATASET.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{DATASET_LOCAL}/", str(DRIVE_DATASET)], check=True)
    print(f"✅ 数据集已成功备份至云盘: {DRIVE_DATASET}")
    hook.update_progress(message="cell3_completed", force_flush=True)

hook.run_stage("cell3_distill_and_generate_dataset", 3, stage3_distill_and_generate_dataset)


## Cell 4：Qwen3-4B / Qwen2.5-3B 基线零样本性能评测 (显式禁用思维链 `enable_thinking=False`)

In [ ]:
import json
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

def stage4_baseline_zero_shot_eval():
    BASE_MODEL_NAME = "Qwen/Qwen3-4B"
    print(f"=== 1. 加载基线模型: {BASE_MODEL_NAME} ===")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    except TypeError:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    base_model.eval()

    val_path = LOCAL_ROUTER / "dataset" / "val.jsonl"
    val_samples = [json.loads(line) for line in open(val_path, encoding="utf-8") if line.strip()]
    eval_subset = val_samples[:100]
    baseline_preds = []

    print(f"\n=== 2. 在验证子集 ({len(eval_subset)} 条) 上执行零样本推理 (禁用思考模式) ===")
    for i, item in enumerate(eval_subset):
        msgs = [
            {"role": "system", "content": item["messages"][0]["content"]},
            {"role": "user", "content": item["messages"][1]["content"]},
        ]
        try:
            prompt = tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
            )

        inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            outputs = base_model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        baseline_preds.append({"prediction": gen_text, "latency_ms": elapsed_ms})

        if (i + 1) % 20 == 0:
            hook.update_progress(step=i + 1, total_steps=len(eval_subset), message="baseline_eval")

    pred_file = LOCAL_ROUTER / "baseline_predictions.jsonl"
    with open(pred_file, "w", encoding="utf-8") as f:
        for p in baseline_preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    EVAL_SCRIPT = REPO_DIR / "training" / "router" / "evaluate_router_offline.py"
    baseline_report_file = LOCAL_ROUTER / "baseline_eval_report.json"
    hook.run_cmd([
        "python", str(EVAL_SCRIPT),
        "--val-file", str(val_path),
        "--predictions", str(pred_file),
        "--max-samples", "100",
        "--output-report", str(baseline_report_file),
    ], stage_name="cell4_baseline_report")

    del base_model
    torch.cuda.empty_cache()
    print("✅ 基线模型评测完毕并已卸载显存。")
    hook.update_progress(message="cell4_completed", force_flush=True)

hook.run_stage("cell4_baseline_zero_shot_eval", 4, stage4_baseline_zero_shot_eval)


## Cell 5：A100 bf16 LoRA SFT 精调 (r=16, lora_alpha=32, Completion-Only 损失)

In [ ]:
import os
import subprocess
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

BASE_MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 1024
OUTPUT_LORA_DIR = LOCAL_ROUTER / "lora_model"

class CompletionPadCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = int(pad_token_id)

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad)
            attention_mask.append(f["attention_mask"] + [0] * pad)
            labels.append(f["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def stage5_qlora_sft():
    print("=== 1. 初始化原生 bfloat16 + SDPA 模型与 LoRA 适配器 ===")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    model.config.use_cache = False
    model.enable_input_require_grads()
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    print("\n=== 2. 格式化训练集与计算 Completion-only 掩码 ===")
    train_data_path = LOCAL_ROUTER / "dataset" / "train.jsonl"
    dataset = load_dataset("json", data_files=str(train_data_path), split="train")

    def tokenize_completion_only(example):
        msgs = example["messages"]
        prompt_msgs = msgs[:2]
        try:
            prompt_str = tokenizer.apply_chat_template(
                prompt_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            full_str = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
        except TypeError:
            prompt_str = tokenizer.apply_chat_template(
                prompt_msgs, tokenize=False, add_generation_prompt=True
            )
            full_str = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
        prompt_ids = tokenizer(prompt_str, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(
            full_str, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LENGTH
        )["input_ids"]
        prompt_len = min(len(prompt_ids), len(full_ids))
        labels = [-100] * prompt_len + full_ids[prompt_len:]
        return {
            "input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": labels,
        }

    tokenized_ds = dataset.map(tokenize_completion_only, remove_columns=dataset.column_names)
    collator = CompletionPadCollator(tokenizer.pad_token_id)

    print("\n=== 3. 启动 A100 bf16 LoRA 训练 ===")
    training_args = TrainingArguments(
        output_dir=str(LOCAL_ROUTER / "checkpoints"),
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=25,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=20,
        optim="adamw_torch_fused",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds,
        data_collator=collator,
        callbacks=[make_hf_trainer_callback(hook)],
    )

    trainer.train()
    print(f"\n保存微调权重至 {OUTPUT_LORA_DIR} ...")
    model.save_pretrained(str(OUTPUT_LORA_DIR))
    tokenizer.save_pretrained(str(OUTPUT_LORA_DIR))

    # 同步断点至 Google Drive
    DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
    DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{LOCAL_ROUTER / 'checkpoints'}/", str(DRIVE_CHECKPOINTS)], check=False)
    print("✅ bf16 LoRA SFT 训练完成并已增量备份至 Google Drive！")
    hook.update_progress(message="cell5_completed", force_flush=True)

hook.run_stage("cell5_qlora_sft", 5, stage5_qlora_sft)


## Cell 6：Post-SFT 验证集评测对比、合并权重并导出 Q4_K_M GGUF 到 Google Drive

In [ ]:
import json
import shutil
import subprocess
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

OUTPUT_LORA_DIR = LOCAL_ROUTER / "lora_model"
MERGED_DIR = LOCAL_ROUTER / "merged_16bit"
DRIVE_EXPORT = DRIVE_ROOT / "export"

def stage6_post_eval_and_gguf_export():
    print("=== 1. 加载微调模型并在验证集上评测 ===")
    tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_LORA_DIR), trust_remote_code=True)
    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-4B",
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    except TypeError:
        base_model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-4B",
            torch_dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    model = PeftModel.from_pretrained(base_model, str(OUTPUT_LORA_DIR))
    model.eval()

    val_path = LOCAL_ROUTER / "dataset" / "val.jsonl"
    val_samples = [json.loads(line) for line in open(val_path, encoding="utf-8") if line.strip()]
    eval_subset = val_samples[:100]
    sft_preds = []

    for i, item in enumerate(eval_subset):
        msgs = [
            {"role": "system", "content": item["messages"][0]["content"]},
            {"role": "user", "content": item["messages"][1]["content"]},
        ]
        try:
            prompt = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        sft_preds.append({"prediction": gen_text, "latency_ms": elapsed_ms})

        if (i + 1) % 20 == 0:
            hook.update_progress(step=i + 1, total_steps=len(eval_subset), message="sft_eval")

    sft_pred_file = LOCAL_ROUTER / "sft_predictions.jsonl"
    with open(sft_pred_file, "w", encoding="utf-8") as f:
        for p in sft_preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    sft_report_file = LOCAL_ROUTER / "export" / "eval_report.json"
    EVAL_SCRIPT = REPO_DIR / "training" / "router" / "evaluate_router_offline.py"
    hook.run_cmd([
        "python", str(EVAL_SCRIPT),
        "--val-file", str(val_path),
        "--predictions", str(sft_pred_file),
        "--max-samples", "100",
        "--output-report", str(sft_report_file),
    ], stage_name="cell6_eval_report")

    sft_eval_summary = {}
    if sft_report_file.is_file():
        try:
            sft_eval_summary = json.loads(sft_report_file.read_text("utf-8"))
        except Exception:
            pass

    print("\n=== 2. 合并 LoRA 权重到 16-bit 模型 ===")
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(str(MERGED_DIR))
    tokenizer.save_pretrained(str(MERGED_DIR))

    del model
    del base_model
    del merged_model
    torch.cuda.empty_cache()

    print("\n=== 3. 转换为 GGUF 并进行 Q4_K_M 量化 ===")
    LLAMA_CPP_DIR = Path("/content/llama.cpp")
    FP16_GGUF = LOCAL_ROUTER / "export" / "dvo-qwen3-4b-router-f16.gguf"
    Q4KM_GGUF = LOCAL_ROUTER / "export" / "dvo-qwen3-4b-router-q4_k_m.gguf"

    hook.run_cmd([
        "python", str(LLAMA_CPP_DIR / "convert_hf_to_gguf.py"),
        str(MERGED_DIR),
        "--outfile", str(FP16_GGUF),
        "--outtype", "f16",
    ], stage_name="cell6_convert_gguf")

    quantize_bin = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"
    if not quantize_bin.is_file():
        quantize_bin = LLAMA_CPP_DIR / "llama-quantize"
    if quantize_bin.is_file():
        hook.run_cmd([str(quantize_bin), str(FP16_GGUF), str(Q4KM_GGUF), "Q4_K_M"], stage_name="cell6_quantize")
        if FP16_GGUF.is_file():
            FP16_GGUF.unlink()  # 节省磁盘空间
    else:
        shutil.copy2(FP16_GGUF, Q4KM_GGUF)

    print("\n=== 4. 拷贝 Modelfile 与评测报告并同步至 Google Drive ===")
    src_modelfile = REPO_DIR / "training" / "router" / "Modelfile.dvo-router"
    if src_modelfile.is_file():
        shutil.copy2(src_modelfile, LOCAL_ROUTER / "export" / "Modelfile.dvo-router")

    DRIVE_EXPORT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{LOCAL_ROUTER / 'export'}/", str(DRIVE_EXPORT)], check=True)

    print(f"\n🎉 路由器全流程交付完成！已同步至云盘:\n   {DRIVE_EXPORT}")
    print("产物包括：")
    print("  1. dvo-qwen3-4b-router-q4_k_m.gguf (高密度量化模型，RTX 4090 推理延迟约 80~110ms)")
    print("  2. Modelfile.dvo-router (Ollama 配置文件)")
    print("  3. eval_report.json (Schema准确率、负样本误报率与延迟评测数据)")
    print("\n在主力机上部署方法：")
    print("  ollama create dvo-router -f Modelfile.dvo-router")
    print("并在 config/local.toml 中配置 router_model = 'dvo-router' 即可生效！")

    hook.finish_job(
        artifacts=[
            DRIVE_EXPORT / "dvo-qwen3-4b-router-q4_k_m.gguf",
            DRIVE_EXPORT / "eval_report.json",
        ],
        summary_metrics=sft_eval_summary,
    )

hook.run_stage("cell6_post_eval_and_gguf_export", 6, stage6_post_eval_and_gguf_export)
